## Introduction to Natural Language Processing with PySpark

### NLP Tools

1. Tokenizer
2. Stop word removal
3. n-grams
4. Term frequency-inverse document frequency (TF-IDF)
5. Count Vectorizer

#### SMS Spam Collection

    * Download the data for later usage

In [1]:
!curl https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip >> smsspamcollection.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  198k    0  198k    0     0  62751      0 --:--:--  0:00:03 --:--:-- 62763


In [2]:
!unzip smsspamcollection.zip

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName('nlp').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/20 22:52:25 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/04/20 22:52:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 22:52:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Tokenizer

In [6]:
from pyspark.ml.feature import Tokenizer, RegexTokenizer
from pyspark.sql.functions import col, udf 
from pyspark.sql.types import IntegerType

In [14]:
sent_df = spark.createDataFrame(
    [
        (0, "Hello I am happy to be learning Apache Spark"),
        (1, "I enjoy learning about python and javascript programming."),
        (2, "I am familiar with machine learning applications"),
        (3, 'here,is,a,list,of,words')
    ], 
    ['id', 'Sentence']
)

In [15]:
sent_df.show()

+---+--------------------+
| id|            Sentence|
+---+--------------------+
|  0|Hello I am happy ...|
|  1|I enjoy learning ...|
|  2|I am familiar wit...|
|  3|here,is,a,list,of...|
+---+--------------------+



In [16]:
tokenizer = Tokenizer(inputCol='sentence', outputCol='words')

regexTokenizer = RegexTokenizer(inputCol='sentence', outputCol='words', pattern='\\W')

countTokens = udf(lambda word: len(word), IntegerType())

tokenized = tokenizer.transform(sent_df)

In [17]:
tokenized.show()

+---+--------------------+--------------------+
| id|            Sentence|               words|
+---+--------------------+--------------------+
|  0|Hello I am happy ...|[hello, i, am, ha...|
|  1|I enjoy learning ...|[i, enjoy, learni...|
|  2|I am familiar wit...|[i, am, familiar,...|
|  3|here,is,a,list,of...|[here,is,a,list,o...|
+---+--------------------+--------------------+



In [18]:
tokenized.select('sentence', 'words').withColumn('tokens', countTokens(col('words'))).show()

+--------------------+--------------------+------+
|            sentence|               words|tokens|
+--------------------+--------------------+------+
|Hello I am happy ...|[hello, i, am, ha...|     9|
|I enjoy learning ...|[i, enjoy, learni...|     8|
|I am familiar wit...|[i, am, familiar,...|     7|
|here,is,a,list,of...|[here,is,a,list,o...|     1|
+--------------------+--------------------+------+



In [19]:
regexTokenized = regexTokenizer.transform(sent_df)
regexTokenized.select('sentence', 'words').withColumn('tokens', countTokens(col('words'))).show()

+--------------------+--------------------+------+
|            sentence|               words|tokens|
+--------------------+--------------------+------+
|Hello I am happy ...|[hello, i, am, ha...|     9|
|I enjoy learning ...|[i, enjoy, learni...|     8|
|I am familiar wit...|[i, am, familiar,...|     7|
|here,is,a,list,of...|[here, is, a, lis...|     6|
+--------------------+--------------------+------+



In [27]:
sent_df_token = regexTokenized.select('sentence', 'words').withColumn('tokens', countTokens(col('words')))

In [31]:
sent_df_token.show()

+--------------------+--------------------+------+
|            sentence|               words|tokens|
+--------------------+--------------------+------+
|Hello I am happy ...|[hello, i, am, ha...|     9|
|I enjoy learning ...|[i, enjoy, learni...|     8|
|I am familiar wit...|[i, am, familiar,...|     7|
|here,is,a,list,of...|[here, is, a, lis...|     6|
+--------------------+--------------------+------+



#### Stop Word Removal

In [20]:
from pyspark.ml.feature import StopWordsRemover

In [23]:
sent_df.show(truncate=False)

+---+---------------------------------------------------------+
|id |Sentence                                                 |
+---+---------------------------------------------------------+
|0  |Hello I am happy to be learning Apache Spark             |
|1  |I enjoy learning about python and javascript programming.|
|2  |I am familiar with machine learning applications         |
|3  |here,is,a,list,of,words                                  |
+---+---------------------------------------------------------+



In [29]:
remover = StopWordsRemover(inputCol='words', outputCol='cleaned')

In [30]:
remover.transform(sent_df_token).show(truncate=False)

+---------------------------------------------------------+-----------------------------------------------------------------+------+--------------------------------------------------+
|sentence                                                 |words                                                            |tokens|cleaned                                           |
+---------------------------------------------------------+-----------------------------------------------------------------+------+--------------------------------------------------+
|Hello I am happy to be learning Apache Spark             |[hello, i, am, happy, to, be, learning, apache, spark]           |9     |[hello, happy, learning, apache, spark]           |
|I enjoy learning about python and javascript programming.|[i, enjoy, learning, about, python, and, javascript, programming]|8     |[enjoy, learning, python, javascript, programming]|
|I am familiar with machine learning applications         |[i, am, familiar, wit

#### N-grams

In [32]:
from pyspark.ml.feature import NGram

In [34]:
sent_df_token.show(truncate=False)

+---------------------------------------------------------+-----------------------------------------------------------------+------+
|sentence                                                 |words                                                            |tokens|
+---------------------------------------------------------+-----------------------------------------------------------------+------+
|Hello I am happy to be learning Apache Spark             |[hello, i, am, happy, to, be, learning, apache, spark]           |9     |
|I enjoy learning about python and javascript programming.|[i, enjoy, learning, about, python, and, javascript, programming]|8     |
|I am familiar with machine learning applications         |[i, am, familiar, with, machine, learning, applications]         |7     |
|here,is,a,list,of,words                                  |[here, is, a, list, of, words]                                   |6     |
+---------------------------------------------------------+----------

In [36]:
bigram = NGram(n=2, inputCol='words', outputCol='bigrams')

In [37]:
bigram_df = bigram.transform(sent_df_token)

In [40]:
bigram_df.select('bigrams').show(truncate=False)

+-----------------------------------------------------------------------------------------------------------+
|bigrams                                                                                                    |
+-----------------------------------------------------------------------------------------------------------+
|[hello i, i am, am happy, happy to, to be, be learning, learning apache, apache spark]                     |
|[i enjoy, enjoy learning, learning about, about python, python and, and javascript, javascript programming]|
|[i am, am familiar, familiar with, with machine, machine learning, learning applications]                  |
|[here is, is a, a list, list of, of words]                                                                 |
+-----------------------------------------------------------------------------------------------------------+



#### Term Frequency - Inverse Document Frequency (TF-IDF)

In [41]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer

In [44]:
sent_df = spark.createDataFrame(
    [
        (0, 0.0, "Hello I am happy to be learning Apache Spark"),
        (1, 0.0, "I enjoy learning about python and javascript programming."),
        (2, 1.0, "I am familiar with machine learning applications"),
        (3, 1.0, 'here,is,a,list,of,words')
    ], 
    ['id', 'label', 'Sentence']
)

In [45]:
sent_df.show(truncate=False)

+---+-----+---------------------------------------------------------+
|id |label|Sentence                                                 |
+---+-----+---------------------------------------------------------+
|0  |0.0  |Hello I am happy to be learning Apache Spark             |
|1  |0.0  |I enjoy learning about python and javascript programming.|
|2  |1.0  |I am familiar with machine learning applications         |
|3  |1.0  |here,is,a,list,of,words                                  |
+---+-----+---------------------------------------------------------+



In [47]:
tokenizer = RegexTokenizer(inputCol='sentence', outputCol='words', pattern='\\W')
words_df = tokenizer.transform(sent_df)

In [48]:
words_df.show(truncate=False)

+---+-----+---------------------------------------------------------+-----------------------------------------------------------------+
|id |label|Sentence                                                 |words                                                            |
+---+-----+---------------------------------------------------------+-----------------------------------------------------------------+
|0  |0.0  |Hello I am happy to be learning Apache Spark             |[hello, i, am, happy, to, be, learning, apache, spark]           |
|1  |0.0  |I enjoy learning about python and javascript programming.|[i, enjoy, learning, about, python, and, javascript, programming]|
|2  |1.0  |I am familiar with machine learning applications         |[i, am, familiar, with, machine, learning, applications]         |
|3  |1.0  |here,is,a,list,of,words                                  |[here, is, a, list, of, words]                                   |
+---+-----+-------------------------------------

In [49]:
hashingTF = HashingTF(inputCol='words', outputCol='rawFeatures', numFeatures=20)
featurized = hashingTF.transform(words_df)

In [50]:
featurized.show(truncate=False)

+---+-----+---------------------------------------------------------+-----------------------------------------------------------------+-----------------------------------------------------------------+
|id |label|Sentence                                                 |words                                                            |rawFeatures                                                      |
+---+-----+---------------------------------------------------------+-----------------------------------------------------------------+-----------------------------------------------------------------+
|0  |0.0  |Hello I am happy to be learning Apache Spark             |[hello, i, am, happy, to, be, learning, apache, spark]           |(20,[3,5,6,7,8,9,12,15,16],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1  |0.0  |I enjoy learning about python and javascript programming.|[i, enjoy, learning, about, python, and, javascript, programming]|(20,[1,5,9,11,12,14,16],[1.0,1.0,1.0,1.0,1.0,1.0,2.0])   

In [51]:
idf = IDF(inputCol='rawFeatures', outputCol='features')
idf_model = idf.fit(featurized)

In [54]:
rescale = idf_model.transform(featurized)
rescale.select('label', 'features').show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|(20,[3,5,6,7,8,9,...|
|  0.0|(20,[1,5,9,11,12,...|
|  1.0|(20,[0,2,5,10,12,...|
|  1.0|(20,[7,8,9,12,15]...|
+-----+--------------------+



#### Count Vectorization

In [55]:
from pyspark.ml.feature import CountVectorizer

In [56]:
df = spark.createDataFrame([
    (0, list('abcde')),
    (1, list('abbbcccdde'))
], ['id', 'words'])

In [58]:
df.show(truncate=False)

+---+------------------------------+
|id |words                         |
+---+------------------------------+
|0  |[a, b, c, d, e]               |
|1  |[a, b, b, b, c, c, c, d, d, e]|
+---+------------------------------+



In [59]:
cv = CountVectorizer(inputCol='words', outputCol='features', vocabSize=5, minDF = 2.0)

In [60]:
model = cv.fit(df)

In [61]:
res = model.transform(df)
res.show(truncate=False)

+---+------------------------------+-------------------------------------+
|id |words                         |features                             |
+---+------------------------------+-------------------------------------+
|0  |[a, b, c, d, e]               |(5,[0,1,2,3,4],[1.0,1.0,1.0,1.0,1.0])|
|1  |[a, b, b, b, c, c, c, d, d, e]|(5,[0,1,2,3,4],[3.0,3.0,2.0,1.0,1.0])|
+---+------------------------------+-------------------------------------+

